# Flatand Graph Data Import Pipeline

## Approach Sketch
To be implemented...
- one node per outgoing edge, label with where it comes from
- look at switches: at fixed distance which 2 are closest -> suggest using GeoPandas API for this.
- add actions as edge attributes (LFR disjointly to all outgoing)
- export enhanced osmnx 
- write importer for enhanced osmnx to GraphRailEnv -> describe what we expect as input, which attributes etc.
- timetable for graph env
- unit test with policy reaching target (possibly hard-coded actions)
- snaphost-visualization (preferrably one osmnx export with railway features -> could be a moving house or roundabout if there is no train feature...)


## Documentation
### OSMnx and GeoPandas API documentation
* https://osmnx.readthedocs.io/en/stable/user-reference.html
* https://github.com/gboeing/osmnx-examples
* https://geopandas.org/en/stable/docs/user_guide.html

### OSM Railway documentation
* https://wiki.openstreetmap.org/wiki/Railways
* https://wiki.openstreetmap.org/wiki/Railway_stations
* https://wiki.openstreetmap.org/wiki/Tag:railway=signal
* https://wiki.openstreetmap.org/wiki/Tag:railway=switch
* https://wiki.openstreetmap.org/wiki/Tag:railway=rail
* https://wiki.openstreetmap.org/wiki/Tag:route%3Drailway


### OpenRailwayMap -> inspect Railway features
* https://openrailwaymap.app/#view=18.86/47.1194931/9.3132586&style=signals

### Insepct OSM by Node ID / Way ID
* https://www.openstreetmap.org/node/2048248546/history/4
* https://www.openstreetmap.org/way/194299504

In [ ]:
# !python -m pip install jupyter osmnx matplotlib  "folium>=0.12" matplotlib mapclassify
import osmnx as ox

ox.settings.log_console = True
ox.__version__

In [ ]:
# ox.settings.overpass_url='http://localhost:12347'
# ox.settings.overpass_rate_limit=False

In [ ]:
place = "Walenstadt"
# tiles = "cartodbpositron"
mk = {"radius": 6}
# cols = ["height", "addr:housenumber", "addr:street", "addr:postcode"]

## Download OSMnx graph

In [ ]:
ox.settings.all_oneway = True
G = ox.graph.graph_from_place(
   place,
   retain_all=False,
   truncate_by_edge=True,
   #simplify=True,
   simplify=False,
   custom_filter='["railway"~"rail"]',
)
fig, ax = ox.plot.plot_graph(G, node_size=0, edge_color="w", edge_linewidth=0.2)

In [ ]:
gdf = ox.features.features_from_place(place, 
                                      tags={"building": True}
                                     )
m = gdf.explore(
    #tiles=tiles,
    #tooltip=cols
)
nodes,edges = ox.convert.graph_to_gdfs(G)

In [ ]:
edges.explore(m=m, color="purple")

In [ ]:
nodes.explore(marker_kwds={"radius": 8})

In [ ]:
list(G.nodes(data=True))

## Visualize networkx graph

In [ ]:
import networkx as nx
nx.draw(G, labels={n[0]: str(n[0]) for n in list(G.nodes(data=True))})

In [ ]:
list(G.successors(2042193127))

In [ ]:
list(G.predecessors(2042193127))

## Stations

In [ ]:
stations = ox.features.features_from_place(place, tags={"railway": "station"})
stations.explore(m=m, 
                 #tiles=tiles, 
                 color="yellow", tooltip="name", marker_kwds=mk)

## Signals

In [ ]:
signals = ox.features.features_from_place(place, tags={"railway": "signal"})

In [ ]:
m = gdf.explore(
    #tiles=tiles,
    #tooltip=cols
)
nodes,edges = ox.convert.graph_to_gdfs(G)
nodes.explore(m=m, color="purple")

## Export graph to graphml

In [ ]:
# TODO must not be simplified - alternative?
ox.io.save_graphml(G, filepath="walensee.graphml")

## Import graph from graphml

In [ ]:
GG = ox.io.load_graphml(filepath="walensee.graphml")

In [ ]:
# TODO MultiDiGraph - due to non-simplify?
GG = nx.DiGraph(GG)

## Graph Rail Env

In [ ]:
#!python -m pip install flatland-rl==4.3.0
from flatland.core.env_observation_builder import DummyObservationBuilder
from flatland.envs.graph.rail_graph_transition_map import GraphTransitionMap
from flatland.core.graph.graph_resource_map import GraphResourceMap
from flatland.envs.graph.distance_map import GraphDistanceMap
from flatland.envs.graph_rail_env import GraphRailEnv
from flatland.envs.timetable_utils import Timetable
from flatland.envs import malfunction_generators as mal_gen
from flatland.envs.rewards import DefaultRewards
from flatland.envs.rail_env import RailEnvActions

In [ ]:
# add actions
for v1,v2, edge_data in GG.edges(data=True):
    # TODO actions on outgoing edge data plus v1's prohibited_actions attribute should partition all actions 
    edge_data["actions"] = {RailEnvActions.MOVE_FORWARD}
    # TODO do we still need it?
    edge_data["straight"] = v2
    #GG.edges(v1,v2)["actions"]= {RailEnvActions.MOVE_FORWARD}
    # print(v1,v2,edge_data)

In [ ]:
GG[2042193127][10287687890]

In [ ]:
GG.get_edge_data(2042193127,10287687890)

In [ ]:
# for v1,v2, edge_data in GG.edges(data=True):
#     print(v1,v2,edge_data)

In [ ]:
resource_map = {}
for n1, n2 in GG.edges:
    resource_map[n1] = str(sorted([n1,n2]))
# print(resource_map)

agent_waypoints = {
    0: [[2042193127], [10287687890]]
}
timetable = Timetable(
    earliest_departures=[[0,None]],
    latest_arrivals=[[None,5]],
    max_episode_steps=99,
)

agent_speeds = {
    0: 1/2
}

graph_env = GraphRailEnv.from_graph(
    g=GG,
    resource_map=resource_map,
    agent_waypoints=agent_waypoints,
    agent_speeds=agent_speeds,
    observation_builder=DummyObservationBuilder(),
    malfunction_generator=mal_gen.NoMalfunctionGen(),
    timetable_generator=lambda *args, **kwargs: timetable,
    seed=42,
    rewards=DefaultRewards(),
)
graph_env.reset()

In [ ]:
GG.get_edge_data(2042193127,10287687890)

In [ ]:
graph_env.rail.g.get_edge_data(2042193127,10287687890)

In [ ]:
graph_env.reset()
for _ in range(99):
    print("==============================")
    print(f"elapsed before step {graph_env._elapsed_steps}")
    for agent in graph_env.agents:
        print(f"agent.initial_configuration={agent.initial_configuration}")
        print(f"agent.current_configuration={agent.current_configuration}")
        print(f"agent.targets={agent.targets}")
        print(f"agent.speed_counter.speed={agent.speed_counter.speed}")
        print(f"agent.speed_counter.max_speed={agent.speed_counter.max_speed}")
        print(f"agent.state={agent.state}")
        
    graph_env.step({0: RailEnvActions.MOVE_FORWARD})
    if graph_env.dones["__all__"]:
        print("==============================")
        print("done")
        break